In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("m0_foundations.ipynb")

# M0 — Foundations and baseline

**TC6035 Part 1 · Milestone 0 · due after Session 1**

By the end of this notebook you will have: your own problem instance, a formal
statement of the optimization problem, a characterization of what makes *your*
instance hard, a demonstration that the two decision layers are genuinely
coupled, and two baselines measured under the protocol you will use for the rest
of the assignment.

> **Budget.** Every run is capped at 5 000 objective evaluations and the cap is
> enforced: the 5 001st call raises `BudgetExceeded`. A batch of `n` candidates
> costs `n`. Where your evaluations went is graded.

> **Carry forward to M1.** Your Monte Carlo baseline median and the nominal power
> you justify in Question 5. M1 is not scored without them.

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

from tc6035 import build_instance, SensorPlacementProblem, BudgetExceeded
from tc6035.runlog import RunSet, save_runset

# Your algorithms live in src/solutions.py so the autograder can re-execute them.
from solutions import ALGORITHMS

STUDENT_ID = "A01234567"   # <-- replace with your own
SALT = 0                   # <-- from the instance manifest

BUDGET = 5_000
N_SEEDS = 30

---
## Question 1 — Your instance

Build the instance belonging to your student ID and report its shape.

In [ ]:
instance = ...

print(f"candidates : {instance.n_candidates}")
print(f"grid points: {instance.n_grid}")
print(f"seed       : {instance.seed}")
print(f"power caps : {instance.power_cap.min():.2f} .. {instance.power_cap.max():.2f}")

In [ ]:
grader.check("q1_instance")

<!-- BEGIN QUESTION -->

---
## Question 2 — Formal statement

State the problem: decision variables and domains, objective, constraint, and
**which infeasibility mechanism you will use** — the penalty returned by
`evaluate`, or repair driven by the `feasible` flag from `evaluate_verbose`.
Justify the choice. It is a design decision and you will be held to it.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — How hard is feasibility?

Sample 2 000 random deployments (active fraction uniform in $[0.1,0.5]$, power
uniform within each site's cap) and compute `feasible_fraction`. Use a separate
problem object with a large budget: this is characterization, not a run.

In [ ]:
probe = SensorPlacementProblem(instance, budget=10**6)
r = np.random.default_rng(1)
n_probe = 2000

feasible = ...
objectives = ...
...
    a = ...
    ...
        a[r.integers(instance.n_candidates)] = ...
    p = ...
    e = ...
    ...
    ...
objectives = ...
feasible_fraction = ...

print(f"feasible fraction : {feasible_fraction:.3f}")
print(f"objective median  : {np.median(objectives):.4f}")

In [ ]:
grader.check("q3_feasibility")

---
## Question 4 — Ruggedness

Estimate ruggedness by the **autocorrelation of a random walk**: from a random
start, take 1 000 single-flip or single-power-nudge steps *accepting every move*,
record the objective, and compute the lag-1 autocorrelation `rho_1`.

Near 1 means neighbours have similar quality; near 0 means the neighbourhood
carries almost no information.

In [ ]:
walk = SensorPlacementProblem(instance, budget=10**6)
r = np.random.default_rng(2)
a = r.random(instance.n_candidates) < 0.3
p = np.minimum(np.full(instance.n_candidates, 12.0), instance.power_cap)

series = ...
...
    ...
        ...
    ...
        j = ...
        p[j] = ...
                       ...
    ...
series = ...
rho_1 = ...

print(f"lag-1 autocorrelation: {rho_1:.4f}")

In [ ]:
grader.check("q4_ruggedness")

---
## Question 5 — Are the layers actually coupled?

The assignment claims the discrete and continuous layers cannot be optimized
independently. **Test that claim on your own instance.**

Implement `two_stage(seed)`: optimize the subset with power frozen at a nominal
value, then optimize power on that fixed subset — each stage getting half the
budget. Implement `joint(seed)`: optimize both together with the full budget.
Run both over `N_SEEDS` **paired** seeds and compare with a Wilcoxon signed-rank
test.

Choose and record `NOMINAL_POWER`. **M1 must cite it.**

In [ ]:
NOMINAL_POWER = ...

def _neighbour(a, p, r, flip_discrete, flip_continuous):
    a2, p2 = a.copy(), p.copy()
    if flip_discrete and (not flip_continuous or r.random() < 0.5):
        a2[r.integers(len(a))] ^= True
    elif flip_continuous:
        j = r.integers(len(p))
        p2[j] = np.clip(p2[j] + r.normal(0, 3.0), instance.spec.power_min, instance.power_cap[j])
    return a2, p2

def _climb(problem, seed, a, p, discrete=True, continuous=True):
    r = np.random.default_rng(seed)
    cur = problem.evaluate(a, p)
    while problem.evaluations_remaining > 0:
        a2, p2 = _neighbour(a, p, r, discrete, continuous)
        v = problem.evaluate(a2, p2)
        if v > cur:
            a, p, cur = a2, p2, v
    return a, p, cur

def _start(seed):
    r = np.random.default_rng(seed + 99)
    a = r.random(instance.n_candidates) < 0.3
    if not a.any():
        a[0] = True
    return a, np.minimum(np.full(instance.n_candidates, NOMINAL_POWER), instance.power_cap)

def joint(seed):
    prob = ...
    a, p = ...
    ...

def two_stage(seed):
    a, p = ...
    s1 = ...
    a, p, _ = ...
    s2 = ...
    ...

joint_scores = ...
staged_scores = ...
W_coupling, p_coupling = ...
coupling_gap = ...

print(f"joint     median {np.median(joint_scores):.4f}")
print(f"two-stage median {np.median(staged_scores):.4f}")
print(f"gap {coupling_gap:+.4f}   joint wins {int((joint_scores > staged_scores).sum())}/{N_SEEDS}")
print(f"Wilcoxon W={W_coupling:.1f}  p={p_coupling:.3e}")

In [ ]:
grader.check("q5_coupling")

---
## Question 6 — Monte Carlo baseline

Implement `monte_carlo(problem, seed, **params)` **in `src/solutions.py`** and
register it in `ALGORITHMS`, then import it here. Uniform sampling until the
budget is exhausted, returning the best objective found.

> Algorithms must live in the module, not only in this notebook: the autograder
> re-executes a sample of your runs and can only do that by calling them. See
> §9 of the assignment specification. Run it over `N_SEEDS` seeds, save the run
logs, and record `MC_BASELINE_MEDIAN`.

**This is the number every later algorithm must beat, and every later milestone
must cite.**

In [ ]:
def monte_carlo(problem, seed):
    r = ...
    n = ...
    best = ...
    ...
        a = ...
        ...
            a[r.integers(n)] = ...
        p = ...
        best = ...
    ...

mc_records, mc_scores = [], []
for seed in range(N_SEEDS):
    prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
    mc_scores.append(monte_carlo(prob, seed))
    mc_records.append(prob.record)
mc_scores = np.array(mc_scores)
MC_BASELINE_MEDIAN = float(np.median(mc_scores))

save_runset("results/m0_monte_carlo.npz",
            RunSet(algorithm="monte_carlo", student_id=STUDENT_ID, params={}, records=mc_records))

print(f"Monte Carlo median {MC_BASELINE_MEDIAN:.4f}   "
      f"IQR [{np.percentile(mc_scores,25):.4f}, {np.percentile(mc_scores,75):.4f}]")

In [ ]:
grader.check("q6_monte_carlo")

---
## Question 7 — Simulated annealing

Implement `simulated_annealing(problem, seed, **params)` **in `src/solutions.py`**,
register it in `ALGORITHMS`, and import it here. Use a schedule **you justify**.

Report the measured acceptance rate of worsening moves at the start and at the
end of a run. A schedule asserted without those two numbers is not justified.

> Note carefully: the convergence-in-probability guarantee for annealing holds
> only under a logarithmic schedule. If you use a geometric one — you should —
> say what you gave up and why.

In [ ]:
T0, ALPHA = ...

def simulated_annealing(problem, seed, t0=T0, alpha=ALPHA):
    r = ...
    n = ...
    a = ...
    ...
        a[r.integers(n)] = ...
    p = ...
    cur = ...
    best = ...
    T = ...
    worse_tried = ...
    early, late = ...
    ...
        a2, p2 = ...
        v = ...
        d = ...
        ...
            take = ...
        ...
            ...
            prob_acc = ...
            take = ...
            ...
            frac = ...
            ...
                ...
            ...
                ...
        ...
            a, p, cur = ...
            best = ...
        ...
    ...

sa_records, sa_scores = [], []
for seed in range(N_SEEDS):
    prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
    score, acc_early, acc_late = simulated_annealing(prob, seed)
    sa_scores.append(score)
    sa_records.append(prob.record)
sa_scores = np.array(sa_scores)

save_runset("results/m0_annealing.npz",
            RunSet(algorithm="simulated_annealing", student_id=STUDENT_ID,
                   params={"t0": T0, "alpha": ALPHA}, records=sa_records))

print(f"SA median {np.median(sa_scores):.4f}   "
      f"IQR [{np.percentile(sa_scores,25):.4f}, {np.percentile(sa_scores,75):.4f}]")
print(f"acceptance of worsening moves: early {acc_early:.3f}  late {acc_late:.3e}")

In [ ]:
grader.check("q7_annealing")

---
## Question 8 — Compare, correctly

Compare simulated annealing against the Monte Carlo baseline over the paired
seeds. Use the test the protocol requires and store the p-value in `p_sa_vs_mc`.

> The runs are paired by seed, the outcomes are not normal, and you are making
> one comparison. Choose accordingly — and be ready to say in your defence why
> a t-test would be the wrong instrument here.

In [ ]:
W_sa, p_sa_vs_mc = ...
effect = ...

print(f"SA vs MC: median paired improvement {effect:+.4f}")
print(f"SA wins {int((sa_scores > mc_scores).sum())}/{N_SEEDS} paired seeds")
print(f"Wilcoxon W={W_sa:.1f}  p={p_sa_vs_mc:.3e}")

In [ ]:
grader.check("q8_comparison")

<!-- BEGIN QUESTION -->

---
## Question 9 — What did you learn about *your* instance?

Write 300–500 words. Address, at minimum:

1. What makes your instance hard, citing your numbers from Q3 and Q4.
2. What the coupling result in Q5 licenses you to claim, and what it does not.
3. *Why* annealing beats uniform sampling here — the mechanism, not the fact.
4. Where your 5 000 evaluations went, and whether that allocation was right.

Vague generalities about metaheuristics score in the Inadequate band. Refer to
your own numbers throughout.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Before you submit

- [ ] `results/` contains both run logs
- [ ] You ran inside the devcontainer (`python scripts/verify_environment.py`)
- [ ] `MC_BASELINE_MEDIAN` and `NOMINAL_POWER` are recorded — **M1 must cite both**
- [ ] Your AI ledger is up to date *as you worked*, not reconstructed
- [ ] Public checks pass: `grader.check_all()`